In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np
from sklearn.impute import SimpleImputer

In [2]:
# Load the data
google_fonts_df = pd.read_csv('google-fonts.csv')

In [3]:
dtype_spec = {
    'Font1': str,
    'Font2': str,
    'Cosine Distance': float,
    'Contrast Level': str
}
font_pairs_df = pd.read_csv('font_pairs_contrast_levels.csv', dtype=dtype_spec)

contrast_mapping = {
    'Similar': 1,
    'Very Similar': 2,
    'Balanced': 3,
    'Moderate': 4,
    'High': 5
}

font_pairs_df['Contrast Level'] = font_pairs_df['Contrast Level'].map(contrast_mapping)

# Verify that mapping is successful and convert missing to median if any
if font_pairs_df['Contrast Level'].isna().any():
    median_value = font_pairs_df['Contrast Level'].median()
    font_pairs_df['Contrast Level'].fillna(median_value, inplace=True)


In [4]:
# Select relevant columns from google-fonts.csv
font_features = google_fonts_df[['family', 'variants_thin', 'variants_thinitalic', 'variants_extralight', 
                                 'variants_extralightitalic', 'variants_light', 'variants_lightitalic', 
                                 'variants_medium', 'variants_mediumitalic', 'variants_semibold', 
                                 'variants_semibolditalic', 'variants_bold', 'variants_bolditalic', 
                                 'variants_extrabold', 'variants_extrabolditalic', 'variants_black', 
                                 'variants_blackitalic', 'variants_italic', 'variants_regular', 
                                 'subsets_adlam', 'subsets_ahom', 'subsets_anatolian-hieroglyphs', 
                                 'subsets_arabic', 'subsets_armenian', 'subsets_avestan', 'subsets_balinese', 
                                 'subsets_bamum', 'subsets_bassa-vah', 'subsets_batak', 'subsets_bengali', 
                                 'subsets_bhaiksuki', 'subsets_brahmi', 'subsets_braille', 'subsets_buginese', 
                                 'subsets_buhid', 'subsets_canadian-aboriginal', 'subsets_carian', 
                                 'subsets_caucasian-albanian', 'subsets_chakma', 'subsets_cham', 'subsets_cherokee', 
                                 'subsets_chinese-hongkong', 'subsets_chinese-simplified', 'subsets_chinese-traditional', 
                                 'subsets_chorasmian', 'subsets_coptic', 'subsets_cuneiform', 'subsets_cypriot', 
                                 'subsets_cypro-minoan', 'subsets_cyrillic', 'subsets_cyrillic-ext', 'subsets_deseret', 
                                 'subsets_devanagari', 'subsets_dogra', 'subsets_duployan', 'subsets_egyptian-hieroglyphs', 
                                 'subsets_elbasan', 'subsets_elymaic', 'subsets_emoji', 'subsets_ethiopic', 
                                 'subsets_georgian', 'subsets_glagolitic', 'subsets_gothic', 'subsets_grantha', 
                                 'subsets_greek', 'subsets_greek-ext', 'subsets_gujarati', 'subsets_gunjala-gondi', 
                                 'subsets_gurmukhi', 'subsets_hanifi-rohingya', 'subsets_hanunoo', 'subsets_hatran', 
                                 'subsets_hebrew', 'subsets_imperial-aramaic', 'subsets_indic-siyaq-numbers', 
                                 'subsets_inscriptional-pahlavi', 'subsets_inscriptional-parthian', 'subsets_japanese', 
                                 'subsets_javanese', 'subsets_kaithi', 'subsets_kannada', 'subsets_kawi', 
                                 'subsets_kayah-li', 'subsets_kharoshthi', 'subsets_khitan-small-script', 'subsets_khmer', 
                                 'subsets_khojki', 'subsets_khudawadi', 'subsets_korean', 'subsets_lao', 'subsets_latin', 
                                 'subsets_latin-ext', 'subsets_lepcha', 'subsets_limbu', 'subsets_linear-a', 'subsets_linear-b', 
                                 'subsets_lisu', 'subsets_lycian', 'subsets_lydian', 'subsets_mahajani', 'subsets_makasar', 
                                 'subsets_malayalam', 'subsets_mandaic', 'subsets_manichaean', 'subsets_marchen', 
                                 'subsets_masaram-gondi', 'subsets_math', 'subsets_mayan-numerals', 'subsets_medefaidrin', 
                                 'subsets_meetei-mayek', 'subsets_mende-kikakui', 'subsets_meroitic', 'subsets_meroitic-cursive', 
                                 'subsets_meroitic-hieroglyphs', 'subsets_miao', 'subsets_modi', 'subsets_mongolian', 
                                 'subsets_mro', 'subsets_multani', 'subsets_music', 'subsets_myanmar', 'subsets_nabataean', 
                                 'subsets_nag-mundari', 'subsets_nandinagari', 'subsets_new-tai-lue', 'subsets_newa', 
                                 'subsets_nko', 'subsets_nushu', 'subsets_nyiakeng-puachue-hmong', 'subsets_ogham', 
                                 'subsets_ol-chiki', 'subsets_old-hungarian', 'subsets_old-italic', 'subsets_old-north-arabian', 
                                 'subsets_old-permic', 'subsets_old-persian', 'subsets_old-sogdian', 'subsets_old-south-arabian', 
                                 'subsets_old-turkic', 'subsets_old-uyghur', 'subsets_oriya', 'subsets_osage', 'subsets_osmanya', 
                                 'subsets_ottoman-siyaq-numbers', 'subsets_pahawh-hmong', 'subsets_palmyrene', 
                                 'subsets_pau-cin-hau', 'subsets_phags-pa', 'subsets_phoenician', 'subsets_psalter-pahlavi', 
                                 'subsets_rejang', 'subsets_runic', 'subsets_samaritan', 'subsets_saurashtra', 'subsets_sharada', 
                                 'subsets_shavian', 'subsets_siddham', 'subsets_signwriting', 'subsets_sinhala', 'subsets_sogdian', 
                                 'subsets_sora-sompeng', 'subsets_soyombo', 'subsets_sundanese', 'subsets_syloti-nagri', 
                                 'subsets_symbols', 'subsets_syriac', 'subsets_tagalog', 'subsets_tagbanwa', 'subsets_tai-le', 
                                 'subsets_tai-tham', 'subsets_tai-viet', 'subsets_takri', 'subsets_tamil', 'subsets_tamil-supplement', 
                                 'subsets_tangsa', 'subsets_tangut', 'subsets_telugu', 'subsets_thaana', 'subsets_thai', 
                                 'subsets_tibetan', 'subsets_tifinagh', 'subsets_tirhuta', 'subsets_toto', 'subsets_ugaritic', 
                                 'subsets_vai', 'subsets_vietnamese', 'subsets_vithkuqi', 'subsets_wancho', 'subsets_warang-citi', 
                                 'subsets_yezidi', 'subsets_yi', 'subsets_zanabazar-square', 'subsets_znamenny', 'category_display', 
                                 'category_handwriting', 'category_monospace', 'category_sans-serif', 'category_serif']]


In [5]:
# Merge font pairs with their font properties
font_pairs_df = font_pairs_df.merge(font_features, left_on='Font1', right_on='family', how='left')
font_pairs_df = font_pairs_df.merge(font_features, left_on='Font2', right_on='family', suffixes=('_1', '_2'), how='left')

In [6]:
# Scaling the features
features = ['Cosine Distance', 'Contrast Level']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(font_pairs_df[features])

In [7]:
# Check for NaNs in the scaled data
print("Checking for NaNs in X_scaled:", np.isnan(X_scaled).any())

Checking for NaNs in X_scaled: False


In [8]:
imputer = SimpleImputer(strategy='mean')  # or 'median', 'most_frequent'
X_scaled_imputed = imputer.fit_transform(X_scaled)

In [9]:
n_clusters = 14  # Number of moods
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
font_pairs_df['cluster'] = kmeans.fit_predict(X_scaled_imputed)


c:\Users\HP\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [10]:
# Check the number of distinct clusters
unique_clusters = len(font_pairs_df['cluster'].unique())
print(f'Number of unique clusters: {unique_clusters}')

Number of unique clusters: 14


In [11]:
# Map clusters to moods
default_mood_mapping = {
    0: 'Casual',
    1: 'Elegant',
    2: 'Modern',
    3: 'Playful',
    4: 'Formal',
    5: 'Retro',
    6: 'Tech',
    7: 'Creative',
    8: 'Vintage',
    9: 'Futuristic',
    10: 'Neutral',
    11: 'Dynamic',
    12: 'Sophisticated',
    13: 'Artistic'
}


In [12]:
mood_mapping = {i: default_mood_mapping[i % len(default_mood_mapping)] for i in range(unique_clusters)}

font_pairs_df['mood'] = font_pairs_df['cluster'].map(mood_mapping)

output_columns = ['Font1', 'Font2', 'Contrast Level', 'Cosine Distance', 'mood']
font_pairs_df.to_csv('font_pairs_with_moods.csv', columns=output_columns, index=False)

print("CSV file 'font_pairs_with_moods.csv' created successfully!")

CSV file 'font_pairs_with_moods.csv' created successfully!
